In [9]:
import pickle
import pandas as pd
input_table = pd.read_csv('./special_marker_result_smooth.csv')
print(input_table.head())

   Unnamed: 0          ra        dec long_name  id_w1  id_w2  special_marker
0           0  359.585847  -1.602909  0000m016    728    704               1
1           1  359.471027  -5.220772  0000m046    812    778               1
2           2    0.167739  -5.683610  0000m061   9439   9154               1
3           3  359.952220 -10.660707  0000m107  27469  26318               1
4           4    0.456357 -10.047728  0000m107  16189  15486               1


In [5]:
# -*- coding: utf-8 -*-
"""
Light Curve Feature Extraction for Multi-modal AI Supplement
Input: MJD and magnitude for two bands, with 7-day alignment tolerance
Output: JSON file with feature names, values, and explanations
"""

import numpy as np
import pandas as pd
from scipy import stats
from scipy.signal import find_peaks
from astropy.timeseries import LombScargle
import json
from datetime import datetime
import os

def align_light_curves(mjd1, mag1, err1, mjd2, mag2, err2, max_time_diff=7.0):
    """
    Align two light curves considering observations within 7 days as simultaneous
    """
    # Convert to numpy arrays and remove NaN values
    mjd1, mag1, err1 = np.array(mjd1), np.array(mag1), np.array(err1)
    mjd2, mag2, err2 = np.array(mjd2), np.array(mag2), np.array(err2)
    
    mask1 = ~(np.isnan(mjd1) | np.isnan(mag1))
    mask2 = ~(np.isnan(mjd2) | np.isnan(mag2))
    
    mjd1, mag1, err1 = mjd1[mask1], mag1[mask1], err1[mask1]
    mjd2, mag2, err2 = mjd2[mask2], mag2[mask2], err2[mask2]
    
    # Find common time points within tolerance
    common_times = []
    mag1_aligned = []
    mag2_aligned = []
    err1_aligned = []
    err2_aligned = []
    
    # For each observation in band1, find closest band2 observation within tolerance
    for i, t1 in enumerate(mjd1):
        time_diffs = np.abs(mjd2 - t1)
        min_idx = np.argmin(time_diffs)
        min_diff = time_diffs[min_idx]
        
        if min_diff <= max_time_diff:
            # Use average time of the two observations
            avg_time = (t1 + mjd2[min_idx]) / 2
            common_times.append(avg_time)
            mag1_aligned.append(mag1[i])
            mag2_aligned.append(mag2[min_idx])
            err1_aligned.append(err1[i])
            err2_aligned.append(err2[min_idx])
    
    # Also check for band2 observations that might not have matched with band1
    for j, t2 in enumerate(mjd2):
        time_diffs = np.abs(mjd1 - t2)
        min_idx = np.argmin(time_diffs)
        min_diff = time_diffs[min_idx]
        
        if min_diff <= max_time_diff:
            avg_time = (t2 + mjd1[min_idx]) / 2
            # Check if this time point already exists (with some tolerance)
            existing_times = np.array(common_times) if common_times else np.array([])
            if len(existing_times) == 0 or np.min(np.abs(existing_times - avg_time)) > 0.1:
                common_times.append(avg_time)
                mag1_aligned.append(mag1[min_idx])
                mag2_aligned.append(mag2[j])
                err1_aligned.append(err1[min_idx])
                err2_aligned.append(err2[j])
    
    # Sort by time
    if common_times:
        sort_idx = np.argsort(common_times)
        common_times = np.array(common_times)[sort_idx]
        mag1_aligned = np.array(mag1_aligned)[sort_idx]
        mag2_aligned = np.array(mag2_aligned)[sort_idx]
        err1_aligned = np.array(err1_aligned)[sort_idx]
        err2_aligned = np.array(err2_aligned)[sort_idx]
    
    return {
        'time': common_times,
        'mag1': mag1_aligned,
        'mag2': mag2_aligned,
        'err1': err1_aligned,
        'err2': err2_aligned
    }

def calculate_sampling_periods(times):
    """
    Calculate dominant sampling periods to identify potential aliasing issues
    """
    if len(times) < 4:
        return {"error": "Insufficient data for sampling analysis"}
    
    # Calculate time differences
    time_diffs = np.diff(times)
    
    # Calculate histogram of time differences to find dominant sampling intervals
    hist, bin_edges = np.histogram(time_diffs, bins=min(10, len(time_diffs)))
    dominant_bin_idx = np.argmax(hist)
    dominant_interval = (bin_edges[dominant_bin_idx] + bin_edges[dominant_bin_idx + 1]) / 2
    
    # Calculate median and mode of sampling intervals
    median_interval = np.median(time_diffs)
    
    # Try to find mode (most common interval)
    unique, counts = np.unique(np.round(time_diffs, 1), return_counts=True)
    mode_interval = unique[np.argmax(counts)] if len(unique) > 0 else median_interval
    
    return {
        "median_sampling_interval": float(median_interval),
        "mode_sampling_interval": float(mode_interval),
        "dominant_histogram_interval": float(dominant_interval),
        "sampling_interval_std": float(np.std(time_diffs)),
        "sampling_regularity": float(1 - (np.std(time_diffs) / median_interval)) if median_interval > 0 else 0
    }

def calculate_essential_features(mjd1, mag1, err1, mjd2, mag2, err2, output_filename):
    """
    Calculate essential light curve features for AI model supplementation
    """
    
    # Align the light curves
    aligned_data = align_light_curves(mjd1, mag1, err1, mjd2, mag2, err2)
    
    # Extract aligned data
    t_aligned = aligned_data['time']
    m1_aligned = aligned_data['mag1']
    m2_aligned = aligned_data['mag2']
    e1_aligned = aligned_data['err1']
    e2_aligned = aligned_data['err2']
    
    # Use original data for single-band features
    t1, m1, e1 = np.array(mjd1), np.array(mag1), np.array(err1)
    t2, m2, e2 = np.array(mjd2), np.array(mag2), np.array(err2)
    
    # Remove NaN values from original data
    mask1 = ~(np.isnan(t1) | np.isnan(m1))
    mask2 = ~(np.isnan(t2) | np.isnan(m2))
    t1, m1, e1 = t1[mask1], m1[mask1], e1[mask1]
    t2, m2, e2 = t2[mask2], m2[mask2], e2[mask2]
    
    features = {
        'metadata': {
            'generated_at': datetime.now().isoformat(),
            'input_filename': output_filename,
            'data_points_band1': len(t1),
            'data_points_band2': len(t2),
            'aligned_data_points': len(t_aligned),
            'description': 'Essential light curve features for multi-modal AI supplementation'
        },
        'features': []
    }
    
    # Helper function to add features with consistent format
    def add_feature(category, name, value, explanation):
        features['features'].append({
            'category': category,
            'name': name,
            'value': value,
            'explanation': explanation
        })
    
    # ===========================================
    # 1. Basic Variability Features
    # ===========================================
    if len(m1) > 0:
        add_feature('variability', 'band1_peak_to_peak_amplitude', 
                   float(np.nanmax(m1) - np.nanmin(m1)),
                   'Total brightness variation range in band 1. Caution: Sparse sampling may underestimate true amplitude.')
        
        add_feature('variability', 'band1_rms_variability', 
                   float(np.sqrt(np.nanmean((m1 - np.nanmean(m1))**2))),
                   'Root mean square variability in band 1. More reliable than peak-to-peak for sparse data.')
        
        add_feature('variability', 'band1_median_absolute_deviation', 
                   float(np.nanmedian(np.abs(m1 - np.nanmedian(m1)))),
                   'Robust measure of variability in band 1. Less sensitive to outliers than RMS.')
        
        if len(m1) > 2:
            # Add sample size warning for higher moments
            skew_exp = f'Asymmetry of brightness distribution in band 1 (positive = right-skewed). '
            skew_exp += f'Estimated from {len(m1)} points. Caution: Higher moments require larger samples for reliability.'
            add_feature('variability', 'band1_skewness', 
                       float(stats.skew(m1)),
                       skew_exp)
            
            kurt_exp = f'Tailedness of brightness distribution in band 1 (positive = heavy tails). '
            kurt_exp += f'Estimated from {len(m1)} points. Caution: Kurtosis estimates are noisy with small samples.'
            add_feature('variability', 'band1_kurtosis', 
                       float(stats.kurtosis(m1)),
                       kurt_exp)
    
    if len(m2) > 0:
        add_feature('variability', 'band2_peak_to_peak_amplitude', 
                   float(np.nanmax(m2) - np.nanmin(m2)),
                   'Total brightness variation range in band 2. Caution: Sparse sampling may underestimate true amplitude.')
        
        add_feature('variability', 'band2_rms_variability', 
                   float(np.sqrt(np.nanmean((m2 - np.nanmean(m2))**2))),
                   'Root mean square variability in band 2. More reliable than peak-to-peak for sparse data.')
    
    # ===========================================
    # 2. Temporal Structure Features
    # ===========================================
    if len(t1) > 1:
        dt1 = np.diff(t1)
        sampling_info = calculate_sampling_periods(t1)
        
        add_feature('temporal', 'band1_median_sampling_interval', 
                   float(np.nanmedian(dt1)),
                   'Typical time between observations in band 1 (days). Half-year sampling common in this dataset.')
        
        add_feature('temporal', 'band1_max_gap', 
                   float(np.nanmax(dt1)),
                   'Largest gap between observations in band 1 (days). Large gaps may miss important variability events.')
        
        add_feature('temporal', 'band1_total_timespan', 
                   float(np.nanmax(t1) - np.nanmin(t1)),
                   'Total duration of observations in band 1 (days). Longer baselines improve period detection.')
        
        if 'median_sampling_interval' in sampling_info:
            add_feature('temporal', 'band1_sampling_regularity', 
                       sampling_info['sampling_regularity'],
                       'Measure of how regular sampling is (1 = perfectly regular). Regular sampling can cause aliasing in period analysis.')
    
    if len(t2) > 1:
        dt2 = np.diff(t2)
        add_feature('temporal', 'band2_median_sampling_interval', 
                   float(np.nanmedian(dt2)),
                   'Typical time between observations in band 2 (days). Half-year sampling common in this dataset.')
    
    # ===========================================
    # 3. Color and Inter-band Features
    # ===========================================
    if len(t_aligned) > 2:
        color = m1_aligned - m2_aligned
        
        color_exp = f'Average color (band1 - band2) across {len(t_aligned)} aligned observations. '
        color_exp += 'Large variations may indicate different physical processes in different bands.'
        add_feature('color', 'mean_color', 
                   float(np.nanmean(color)),
                   color_exp)
        
        add_feature('color', 'color_variability', 
                   float(np.nanstd(color)),
                   'Standard deviation of color variations. Stable colors suggest similar physical origins across bands.')
        
        add_feature('color', 'color_amplitude', 
                   float(np.nanmax(color) - np.nanmin(color)),
                   'Total range of color variations. Large amplitudes may indicate spectral changes.')
        
        # Correlation between bands
        if len(m1_aligned) > 1:
            corr_coef = np.corrcoef(m1_aligned, m2_aligned)[0, 1]
            corr_exp = f'Pearson correlation between bands based on {len(t_aligned)} aligned points. '
            corr_exp += 'High correlation suggests coordinated variability across bands.'
            add_feature('color', 'band_correlation', 
                       float(corr_coef),
                       corr_exp)
    
    # ===========================================
    # 4. Trend Features
    # ===========================================
    if len(t1) > 2 and len(m1) > 2:
        # Linear trend
        valid_mask = ~(np.isnan(t1) | np.isnan(m1))
        if np.sum(valid_mask) > 2:
            slope, intercept, r_value, p_value, std_err = stats.linregress(
                t1[valid_mask], m1[valid_mask])
            
            trend_exp = f'Slope of linear fit (mag/day), positive = fading. '
            trend_exp += f'Based on {np.sum(valid_mask)} data points. Caution: Sparse sampling may not capture complex trends.'
            add_feature('trend', 'band1_linear_trend_slope', 
                       float(slope),
                       trend_exp)
            
            sig_exp = f'P-value for linear trend significance. Small values (<0.05) indicate significant trend. '
            sig_exp += f'Note: Sparse data reduces statistical power for trend detection.'
            add_feature('trend', 'band1_trend_significance', 
                       float(p_value),
                       sig_exp)
            
            change_exp = f'Total brightness change over {float(np.nanmax(t1) - np.nanmin(t1)):.1f} days based on linear trend. '
            change_exp += 'Extrapolation beyond data range is uncertain.'
            add_feature('trend', 'band1_total_brightness_change', 
                       float(slope * (np.nanmax(t1) - np.nanmin(t1))),
                       change_exp)
    
    # ===========================================
    # 5. Periodicity Features (Enhanced with sampling analysis)
    # ===========================================
    if len(t1) > 10:
        try:
            # Remove NaN values for periodogram
            mask = ~(np.isnan(t1) | np.isnan(m1))
            t_clean, m_clean = t1[mask], m1[mask]
            
            if len(t_clean) > 10:
                # Calculate sampling properties for period analysis context
                sampling_info = calculate_sampling_periods(t_clean)
                
                # Calculate periodogram with wider frequency range
                frequency, power = LombScargle(t_clean, m_clean).autopower(
                    minimum_frequency=1/2000, 
                    maximum_frequency=1/180  # Higher maximum to detect shorter periods
                )
                
                if len(power) > 0:
                    best_idx = np.argmax(power)
                    best_period = 1 / frequency[best_idx]
                    max_power = power[best_idx]
                    
                    # Calculate false alarm probability
                    fap = LombScargle(t_clean, m_clean).false_alarm_probability(max_power)
                    
                    # Check if best period is near sampling intervals (potential aliasing)
                    sampling_alias_warning = ""
                    if 'median_sampling_interval' in sampling_info:
                        sampling_interval = sampling_info['median_sampling_interval']
                        # Check if period is close to sampling interval or its multiples
                        for mult in [0.5, 1, 2, 3]:
                            alias_period = sampling_interval * mult
                            if 0.8 * alias_period <= best_period <= 1.2 * alias_period:
                                sampling_alias_warning = f" WARNING: Detected period ({best_period:.1f} days) is close to {mult}x sampling interval ({alias_period:.1f} days) - may be aliasing artifact."
                                break
                    
                    period_exp = f'Most significant period detected. Based on {len(t_clean)} data points. '
                    period_exp += f'Minimum {15-20} points recommended for reliable period detection.{sampling_alias_warning}'
                    add_feature('periodicity', 'dominant_period', 
                               float(best_period),
                               period_exp)
                    
                    power_exp = f'Lomb-Scargle power at dominant period. Higher values indicate stronger periodicity. '
                    power_exp += f'Note: Sparse data ({len(t_clean)} points) reduces period detection sensitivity.'
                    add_feature('periodicity', 'period_power', 
                               float(max_power),
                               power_exp)
                    
                    fap_exp = f'Probability that period is due to random noise. Values <0.01 indicate significant detection. '
                    fap_exp += f'Caution: FAP estimates are less reliable with sparse data.'
                    add_feature('periodicity', 'false_alarm_probability', 
                               float(fap),
                               fap_exp)
                    
                    add_feature('periodicity', 'period_significant', 
                               bool(fap < 0.01),
                               'True if period is statistically significant (FAP < 1%). Consider data sparsity when interpreting.')
                    
                    # Add sampling information to periodicity context
                    if 'median_sampling_interval' in sampling_info:
                        sampling_exp = f'Median sampling interval. Periods near this value or its multiples may be aliasing artifacts. '
                        sampling_exp += f'Regular sampling (regularity={sampling_info["sampling_regularity"]:.2f}) increases aliasing risk.'
                        add_feature('periodicity', 'median_sampling_interval', 
                                   sampling_info['median_sampling_interval'],
                                   sampling_exp)
                    
                    # Find secondary peaks in periodogram
                    if len(power) > 10:
                        # Find peaks excluding the highest one
                        peaks, properties = find_peaks(power, height=np.median(power)*1.5, distance=5)
                        if len(peaks) > 1:
                            # Sort peaks by power (descending) and take second highest
                            sorted_peaks = peaks[np.argsort(power[peaks])[::-1]]
                            if len(sorted_peaks) > 1 and sorted_peaks[0] == best_idx:
                                second_best_idx = sorted_peaks[1]
                                second_period = 1 / frequency[second_best_idx]
                                second_power = power[second_best_idx]
                                
                                second_exp = f'Second most significant period. Useful for identifying harmonic relationships. '
                                second_exp += f'Based on {len(t_clean)} points - interpret with caution due to sparse sampling.'
                                add_feature('periodicity', 'secondary_period', 
                                           float(second_period),
                                           second_exp)
        except Exception as e:
            error_exp = f'Period analysis failed: {str(e)}. Common with very sparse or noisy data.'
            add_feature('periodicity', 'error', str(e), error_exp)
    else:
        skip_exp = f'Period analysis skipped: only {len(t1)} data points available. '
        skip_exp += 'Minimum 10-15 points recommended, with 20+ preferred for reliable period detection.'
        add_feature('periodicity', 'period_analysis_skipped', 
                   None, skip_exp)
    
    # ===========================================
    # 6. Morphological Features (Adjusted for sparse data)
    # ===========================================
    if len(m1) > 5:
        # Peak detection with adjusted parameters for sparse data
        try:
            # Use lower prominence threshold for sparse data
            prominence_threshold = 0.5 * np.std(m1) if len(m1) < 15 else 0.1
            peaks, properties = find_peaks(-m1, prominence=prominence_threshold)
            valleys, _ = find_peaks(m1, prominence=prominence_threshold)
            
            peaks_exp = f'Number of significant brightness peaks detected. '
            peaks_exp += f'Using relaxed threshold for {len(m1)}-point sparse data. May include noise fluctuations.'
            add_feature('morphology', 'number_of_peaks', 
                       int(len(peaks)),
                       peaks_exp)
            
            valleys_exp = f'Number of significant brightness valleys detected. '
            valleys_exp += f'Using relaxed threshold for {len(m1)}-point sparse data. Interpret with caution.'
            add_feature('morphology', 'number_of_valleys', 
                       int(len(valleys)),
                       valleys_exp)
            
            # Simple symmetry measure (if we have both peaks and valleys)
            if len(peaks) > 0 and len(valleys) > 0:
                peak_heights = -m1[peaks]  # Convert back to magnitude scale
                valley_heights = m1[valleys]
                symmetry = np.std(peak_heights) / (np.std(peak_heights) + np.std(valley_heights))
                symmetry_exp = f'Measure of symmetry between peaks and valleys (0.5 = symmetric). '
                symmetry_exp += f'Based on {len(peaks)} peaks and {len(valleys)} valleys from sparse data - uncertain.'
                add_feature('morphology', 'lightcurve_symmetry', 
                           float(symmetry),
                           symmetry_exp)
        except Exception as e:
            add_feature('morphology', 'error', str(e),
                       f'Peak detection failed for sparse data ({len(m1)} points): {str(e)}')
    
    # ===========================================
    # 7. Data Quality Features
    # ===========================================
    if len(e1) > 0:
        add_feature('quality', 'band1_mean_error', 
                   float(np.nanmean(e1)),
                   'Average photometric error in band 1. Larger errors reduce feature reliability.')
    
    if len(e2) > 0:
        add_feature('quality', 'band2_mean_error', 
                   float(np.nanmean(e2)),
                   'Average photometric error in band 2. Larger errors reduce feature reliability.')
    
    aligned_frac = len(t_aligned) / max(len(t1), len(t2), 1)
    aligned_exp = f'Fraction of observations that could be aligned between bands. '
    aligned_exp += f'Low values ({aligned_frac:.2f}) reduce reliability of color and correlation features.'
    add_feature('quality', 'aligned_fraction', 
               float(aligned_frac),
               aligned_exp)
    
    # Add overall data quality assessment
    total_points = len(t1) + len(t2)
    quality_note = ""
    if total_points < 20:
        quality_note = "VERY SPARSE: Features have high uncertainty"
    elif total_points < 40:
        quality_note = "SPARSE: Interpret features with caution"
    elif total_points < 100:
        quality_note = "MODERATE: Reasonable feature reliability"
    else:
        quality_note = "GOOD: Features should be reliable"
    
    add_feature('quality', 'data_quality_note', 
               quality_note,
               f'Overall data quality assessment based on {total_points} total observations. Sparse data (<20 points) reduces reliability.')
    
    return features

def save_features_to_json(features_dict, output_filename):
    """
    Save features to JSON file with proper formatting
    """
    # Ensure output directory exists
    os.makedirs(os.path.dirname(output_filename) if os.path.dirname(output_filename) else '.', 
                exist_ok=True)
    
    # Add .json extension if not present
    if not output_filename.endswith('.json'):
        output_filename += '.json'
    
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(features_dict, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Features saved to: {output_filename}")
    print(f"📊 Total features calculated: {len(features_dict['features'])}")
    
    # Print feature categories
    categories = set(feat['category'] for feat in features_dict['features'])
    print(f"📋 Feature categories: {', '.join(categories)}")

# ===========================================
# Example Usage
# ===========================================
# if __name__ == "__main__":
#     # Example data with sparse sampling (similar to your dataset)
#     np.random.seed(42)
    
#     # Band 1 data (sparse sampling, ~15 points)
#     t1 = np.sort(np.concatenate([
#         np.linspace(0, 200, 5),
#         np.linspace(400, 600, 5), 
#         np.linspace(800, 1000, 5)
#     ]))
#     m1 = 15 + 0.5 * np.sin(t1/100) + np.random.normal(0, 0.1, len(t1))
#     e1 = np.full_like(m1, 0.1)
    
#     # Band 2 data (different sparse sampling)
#     t2 = np.sort(np.concatenate([
#         np.linspace(50, 250, 5),
#         np.linspace(450, 650, 4),
#         np.linspace(850, 1050, 4)
#     ]))
#     m2 = 16 + 0.3 * np.sin(t2/100) + np.random.normal(0, 0.15, len(t2))
#     e2 = np.full_like(m2, 0.15)
    
#     # Calculate features
#     output_name = "sparse_lightcurve_features"
#     features = calculate_essential_features(t1, m1, e1, t2, m2, e2, output_name)
    
#     # Save to JSON
#     save_features_to_json(features, output_name)

In [10]:
import json
from util import make_single_light_curve
for i in range(len(input_table)):
    long_name = input_table.iloc[i]['long_name']
    name = long_name[0:3]
    id_w1 = input_table.iloc[i]['id_w1']
    id_w2 = input_table.iloc[i]['id_w2']
    table1 = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_w1_mached.csv')
    table2 = pd.read_csv('./mached_catalog/'+name+'/'+long_name+'/'+long_name+'_w2_mached.csv')
    ra1,dec1,mag1,error1,mjdmean1 = make_single_light_curve(table1,int(id_w1-1))
    ra2,dec2,mag2,error2,mjdmean2 = make_single_light_curve(table2,int(id_w2-1))
    output_name = './special_feature_files/%s_%d_%d.json' % (long_name, id_w1, id_w2)
    features = calculate_essential_features(np.array(mjdmean1,dtype='float'), np.array(mag1,dtype='float'), np.array(error1,dtype='float'),
                                            np.array(mjdmean2,dtype='float'), np.array(mag2,dtype='float'), np.array(error2,dtype='float'),
                                             output_name)
    save_features_to_json(features, output_name)
    print(i)

✅ Features saved to: ./special_feature_files/0000m016_728_704.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
0
✅ Features saved to: ./special_feature_files/0000m046_812_778.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
1
✅ Features saved to: ./special_feature_files/0000m061_9439_9154.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
2
✅ Features saved to: ./special_feature_files/0000m107_27469_26318.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
3


/tmp/ipykernel_97392/2900720379.py:410: RuntimeWarning: invalid value encountered in double_scalars
  symmetry = np.std(peak_heights) / (np.std(peak_heights) + np.std(valley_heights))


✅ Features saved to: ./special_feature_files/0000m107_16189_15486.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
4
✅ Features saved to: ./special_feature_files/0000m167_9224_8878.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
5
✅ Features saved to: ./special_feature_files/0000m182_13559_12955.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
6
✅ Features saved to: ./special_feature_files/0000m288_18453_17727.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
7
✅ Features saved to: ./special_feature_files/0000m349_28938_27376.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
8
✅ Features saved 

/tmp/ipykernel_97392/2900720379.py:410: RuntimeWarning: invalid value encountered in double_scalars
  symmetry = np.std(peak_heights) / (np.std(peak_heights) + np.std(valley_heights))


✅ Features saved to: ./special_feature_files/0000m425_26058_25062.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
10


...

✅ Features saved to: ./special_feature_files/1409p090_7303_7022.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
92


/tmp/ipykernel_97392/2900720379.py:410: RuntimeWarning: invalid value encountered in double_scalars
  symmetry = np.std(peak_heights) / (np.std(peak_heights) + np.std(valley_heights))


✅ Features saved to: ./special_feature_files/1410m667_27713_27734.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
93
✅ Features saved to: ./special_feature_files/1410m667_45255_45281.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
94


/tmp/ipykernel_97392/2900720379.py:410: RuntimeWarning: invalid value encountered in double_scalars
  symmetry = np.std(peak_heights) / (np.std(peak_heights) + np.std(valley_heights))


✅ Features saved to: ./special_feature_files/1411p590_7059_6788.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
95
✅ Features saved to: ./special_feature_files/1412p045_9597_9267.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
96
✅ Features saved to: ./special_feature_files/1413m561_38118_38199.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
97
✅ Features saved to: ./special_feature_files/1415m107_16556_15930.json
📊 Total features calculated: 32
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
98
✅ Features saved to: ./special_feature_files/1415m440_14592_14598.json
📊 Total features calculated: 31
📋 Feature categories: morphology, periodicity, color, variability, temporal, trend, quality
99


In [32]:
# with open('special_lc_feature_description.json','w') as f:
#     json.dump(all_features_description,f)